In [ ]:
import sys; sys.path.append('..')
import MeshFEM
import sheet_convergence, sim_utils, py_newton_optimizer
import elastic_solid, energy, sim_utils, mesh
from viewer import Viewer
import numpy as np

from matplotlib import pyplot as plt

In [ ]:
STRETCH_MAGNITUDE = 0.1
def meshForResolution(maxArea, deg=1):
    return sheet_convergence.getMesh(maxArea, 1, degree=deg, embeddingDimension=2)
    
def equilibriumForResolution(maxArea = 1, deg=1, psi = energy.IsotropicLinearElastic(2, 1, -0.3)):
    m = meshForResolution(maxArea, deg=deg)
    es = elastic_solid.ElasticSolid(m, psi)

    rightBCVars = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MAX_X, displacementComponents=[0])
    leftBCVars  = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MIN_X)

    es.setIdentityDeformation()
    x = es.getVars()
    x[rightBCVars] *= 1.0 + STRETCH_MAGNITUDE
    es.setVars(x)

    opts = py_newton_optimizer.NewtonOptimizerOptions()
    opts.verbose = 0
    es.computeEquilibrium(fixedVars=rightBCVars + leftBCVars, opts=opts)
    return es

In [ ]:
maxAreas = np.logspace(-2, -5.75, 20)
es_d1 = [equilibriumForResolution(ma) for ma in maxAreas]
es_d2 = [equilibriumForResolution(ma, deg=2) for ma in maxAreas]

In [ ]:
medianEdgeLens = [np.median(es.mesh().edgeLengths()) for es in es_d1]

In [ ]:
es_gt = es_d2[-1]
m_gt = es_gt.mesh()
x_gt = es_gt.getDeformedPositions()

In [ ]:
energy_gt = es_gt.energy()
def energy_error(es): return np.abs(es.energy() - energy_gt) / energy_gt

In [ ]:
plt.loglog(medianEdgeLens[:-1], [energy_error(es) for es in es_d1[:-1]])
plt.loglog(medianEdgeLens[:-1], [energy_error(es) for es in es_d2[:-1]])
plt.xlabel('Median h')
plt.ylabel('Energy Error')
plt.grid()

In [ ]:
import field_sampler
def subsampledSolution(es): return field_sampler.FieldSampler(es.mesh()).sample(m_gt.nodes(), es.getDeformedPositions())
def euclideanSolnRelError(es): return np.linalg.norm(subsampledSolution(es) - es_gt.getDeformedPositions()) / np.linalg.norm(x_gt)

In [ ]:
plt.loglog(medianEdgeLens[:-1], [euclideanSolnRelError(es) for es in es_d1[:-1]])
plt.loglog(medianEdgeLens[:-1], [euclideanSolnRelError(es) for es in es_d2[:-1]])
plt.xlabel('Median h')
plt.ylabel('Deformation Error')
plt.grid()

# Subdivsion version

In [ ]:
import igl

In [ ]:
def equilibriumForSubdiv(nsubdiv = 0, deg=1, psi = energy.IsotropicLinearElastic(2, 1, 0.3)):
    m = sheet_convergence.getMesh(1, 1, embeddingDimension=2)
    V, F = m.vertices(), m.elements()
    
    #for i in range(nsubdiv):
    V, F = igl.upsample(V, F, number_of_subdivs=nsubdiv)
    m = mesh.Mesh(V, F, degree=deg, embeddingDimension=2)
        
    es = elastic_solid.ElasticSolid(m, psi)

    rightBCVars = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MAX_X, displacementComponents=[0])
    leftBCVars  = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MIN_X)

    es.setIdentityDeformation()
    x = es.getVars()
    x[rightBCVars] *= 1.0 + STRETCH_MAGNITUDE
    es.setVars(x)

    opts = py_newton_optimizer.NewtonOptimizerOptions()
    opts.verbose = 0
    es.computeEquilibrium(fixedVars=rightBCVars + leftBCVars, opts=opts)
    return es

In [ ]:
def visualizeForSubdiv(nsubdiv = 0, deg=1, psi = energy.IsotropicLinearElastic(2, 1, 0.4)):
    es = equilibriumForSubdiv(nsubdiv, deg=deg, psi=psi)
    return Viewer(es, wireframe=True).antialiasedImage()

In [ ]:
visualizeForSubdiv(7)

In [ ]:
nsubdivs = np.arange(10)

In [ ]:
es_sub_d1 = [equilibriumForSubdiv(nsubdiv) for nsubdiv in nsubdivs]
es_sub_d2 = [equilibriumForSubdiv(nsubdiv, deg=2) for nsubdiv in nsubdivs]

In [ ]:
es_sub_gt = es_sub_d2[-1].energy()
def energy_error_sub(es): return np.abs(es.energy() - es_sub_gt) / es_sub_gt

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
plt.loglog(np.power(2.0, -nsubdivs[:-1]), [energy_error_sub(es) for es in es_sub_d1[:-1]])
plt.loglog(np.power(2.0, -nsubdivs[:-1]), [energy_error_sub(es) for es in es_sub_d2[:-1]])
plt.grid()
#plt.axes('square')